# Análise de Sensibilidade: Da Geometria ao Computador

**Pesquisa Operacional I · Análise de Sensibilidade**  
**Aluno:** Matheus Sousa Marinho &nbsp;·&nbsp; **Matrícula:** 202206132

---

## Estrutura do notebook

| Parte | Problema | Tarefas |
|-------|----------|--------|
| 1 | **Toyco** (Taha §3.6) | 1.1 Executar e validar · 1.2 Decisão gerencial |
| 2 | **NeuralCloud Bruto** | 2.1 Adaptação computacional · 2.2 Melhor investimento |
| 3 | **NeuralCloud Líquido** | 2.3 Fragilidade do ótimo |


## 0. Setup

Instala `amplpy`, baixa o CPLEX e inicializa o objeto `ampl`. Rode **uma vez** por sessão.

In [1]:
!pip install -q amplpy
from amplpy import AMPL, ampl_notebook
import pandas as pd

ampl = ampl_notebook(
    modules=["cplex"],
    license_uuid="default",
)

AMPL Version 20260520 (Linux-6.8.0-1052-azure, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/home/m9t/Documents/ufg/o-research/09/.venv/lib/python3.12/site-packages/ampl_module_base/bin/ampl.lic".



---
## Parte 1 — Toyco

**Referência:** A Toyco monta trens ($x_1$), caminhões ($x_2$) e carros ($x_3$) em três operações.  
Receitas: \$3, \$2, \$5. Capacidades: 430, 460, 420 min/dia.

### 1.1 Modelo AMPL (`toyco.mod`)

Estrutura indexada: `x[j]` com `j ∈ PROD` e restrições `Capacidade {i ∈ OP}` — o mesmo padrão para o NeuralCloud.

In [2]:
%%writefile toyco.mod
# ---- Toyco: análise de sensibilidade (Taha §3.6) ----
set OP;       # operações  (Op1, Op2, Op3)
set PROD;     # produtos   (Trem, Caminhao, Carro)

param margem {PROD} >= 0;      # receita por unidade ($)
param tempo  {OP, PROD} >= 0;  # min de operação i por unidade do produto j
param cap    {OP} >= 0;        # capacidade diária de cada operação (min/dia)

var x {PROD} >= 0;

maximize z: sum {j in PROD} margem[j] * x[j];

s.t. Capacidade {i in OP}:
    sum {j in PROD} tempo[i,j] * x[j] <= cap[i];

Writing toyco.mod


### 1.2 Dados (`toyco.dat`)

In [3]:
%%writefile toyco.dat
set OP   := Op1 Op2 Op3 ;
set PROD := Trem Caminhao Carro ;

param margem :=
    Trem      3
    Caminhao  2
    Carro     5 ;

param tempo : Trem  Caminhao  Carro :=
    Op1        1      2        1
    Op2        3      0        2
    Op3        1      4        0 ;

param cap :=
    Op1  430
    Op2  460
    Op3  420 ;

Writing toyco.dat


### 1.3 Resolver com `sens=1`

In [4]:
ampl.reset()
ampl.read("toyco.mod")
ampl.read_data("toyco.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z = ampl.get_objective("z").value()
print(f"z* = $ {z:,.2f}")

x_star = ampl.get_variable("x").get_values().to_pandas()
x_star.columns = ["x*"]
print("\nAlocação ótima:")
display(x_star)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 1350
3 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* = $ 1,350.00

Alocação ótima:


,x*
Caminhao,100
Carro,230
Trem,0


### 1.4 Tabela de Restrições — preço-sombra e faixa de viabilidade

| coluna | atributo AMPL | significado |
|--------|--------------|-------------|
| `b` | `Capacidade.ub` | RHS atual ($b_i$) |
| `y` | `Capacidade.dual` | preço-sombra $y_i$ |
| `rhslo`/`rhshi` | `Capacidade.sensrhslo/hi` | faixa de viabilidade |

In [5]:
df_restr = ampl.get_data(
    "Capacidade.body",
    "Capacidade.ub",
    "Capacidade.dual",
    "Capacidade.sensrhslo",
    "Capacidade.sensrhshi",
).to_pandas()
df_restr.columns = ["body", "b", "y", "rhslo", "rhshi"]
df_restr["folga"]  = df_restr["b"] - df_restr["body"]
df_restr["status"] = df_restr["folga"].apply(
    lambda f: "ativa" if abs(f) < 1e-6 else "folgada"
)
df_restr = df_restr[["b", "folga", "y", "rhslo", "rhshi", "status"]]
df_restr = df_restr.sort_values("y", ascending=False)
display(df_restr.round(4))

,b,folga,y,rhslo,rhshi,status
Op2,460,0,2,440,860,ativa
Op1,430,0,1,230,440,ativa
Op3,420,20,0,400,100000000000000000000,folgada


### 1.5 Tabela de Variáveis — custo reduzido e faixa de otimalidade

In [6]:
df_var = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_var.columns = ["x*", "rc", "objlo", "objhi"]

margem = ampl.get_parameter("margem").to_pandas()
margem.columns = ["c_j"]

df_var = df_var.join(margem)
df_var = df_var[["c_j", "x*", "rc", "objlo", "objhi"]]
display(df_var.round(4))

,c_j,x*,rc,objlo,objhi
Caminhao,2,100,0,0,10
Carro,5,230,0,2.333333,100000000000000000000
Trem,3,0,-4,-100000000000000000000,7


### 1.6 Tarefa 1.2 — Decisão Gerencial

Use **somente** as tabelas acima, sem rodar o modelo de novo.  
Em cada item aplique a **Regra dos Três Elementos**: (1) $y_i$ ou `rc` · (2) variação · (3) faixa.

#### (a) Vale a pena alugar capacidade extra da Op.1 a \$0,80/min?

1. **Preço-sombra:** $y_1 = 1{,}00$ \$/min — cada minuto adicional na Op.1 gera \$1,00 a mais em $z$.
2. **Variação proposta:** alugar $\Delta b_1$ minutos extras a \$0,80/min.
3. **Faixa de validade:** $y_1$ permanece válido enquanto $b_1 \leq 440$ min; como $b_1 = 430$, há margem de até **10 min adicionais**.

**Decisão: sim, vale a pena.** O ganho por minuto alugado (\$1,00) supera o custo (\$0,80), gerando lucro líquido de **\$0,20/min**. Recomenda-se alugar até o limite de 10 min extras (até $b_1 = 440$); acima disso, $y_1$ pode mudar e a análise precisa ser refeita.

#### (b) A gerência sugere subir a Op.2 de 460 para 600 min. Qual o ganho previsto em $z$? E se subir para 900 min?

**Para 600 min:**

1. **Preço-sombra:** $y_2 = 2{,}00$ \$/min.
2. **Variação proposta:** $\Delta b_2 = 600 - 460 = 140$ min.
3. **Faixa de validade:** $y_2$ é válido enquanto $b_2 \leq 860$ min; como $600 \leq 860$, a variação está **dentro da faixa**.

**Ganho previsto:** $\Delta z = y_2 \cdot \Delta b_2 = 2{,}00 \times 140 = \mathbf{\$280}$.

---

**Para 900 min:**

1. **Preço-sombra:** $y_2 = 2{,}00$ \$/min (válido somente até $b_2 = 860$ min).
2. **Variação proposta:** $\Delta b_2 = 900 - 460 = 440$ min (variação pretendida).
3. **Faixa de validade:** margem disponível $= 860 - 460 = 400$ min; como $900 > 860$, a expansão **ultrapassa a faixa**.

**Conclusão:** O preço-sombra $y_2 = 2{,}00$ só garante $\Delta z = 2{,}00 \times 400 = \mathbf{\$800}$ até o limite de 860 min. Entre 860 e 900 min a base ótima muda e $y_2$ assume um novo valor — sem reotimização não é possível prever o ganho nesse intervalo.

#### (c) Por que aumentar a Op.3 não muda $z$, embora tenha folga de apenas 20 min?

1. **Preço-sombra:** $y_3 = 0$ — Op.3 é uma restrição **folgada** (não-ativa): a solução ótima usa apenas 400 dos 420 min disponíveis.
2. **Variação proposta:** $\Delta b_3 > 0$ (qualquer expansão da Op.3).
3. **Faixa de validade:** $y_3 = 0$ é válido para todo $b_3 \geq 400$; a restrição só se tornaria ativa — e $y_3$ passaria a ser positivo — se a capacidade fosse **reduzida** abaixo de 400 min.

**Conclusão:** o que limita a produção são Op.1 ($y_1 = 1$) e Op.2 ($y_2 = 2$), não a Op.3. Como já sobram 20 min na Op.3, ela não é gargalo; adicionar capacidade onde não há gargalo não libera produção extra e, portanto, não altera $z$. O tamanho da folga (20 min) é irrelevante — o que importa é que existe folga, pois isso garante $y_3 = 0$.

---
## Parte 2 — NeuralCloud Bruto

**Contexto:** A NeuralCloud vende planos de GPU (Basic, Pro, Ultra) em três datacenters (DC1, DC2, DC3).  
O modelo **bruto** maximiza a receita bruta — sem descontar custos operacionais.

| | Toyco | NeuralCloud |
|--|--|--|
| Conjuntos | `OP`, `PROD` | `DC`, `PLANO` |
| Variáveis | `var x {PROD}` (1 índice) | `var x {DC, PLANO}` (2 índices) |
| Restrições | `Capacidade {OP}` | `Capac {DC}`, `Potencia {DC}`, `Demanda {PLANO}` |

> **Tarefa 2.1:** Adapte o padrão de extração da Toyco para lidar com variáveis duplamente indexadas e múltiplas famílias de restrição.

### 2.1 Modelo (`bruto.mod`)

Três famílias de restrição:
- **`Capac {DC}`** — limite de slots GPU por datacenter
- **`Potencia {DC}`** — limite de potência elétrica (kW) por datacenter  
- **`Demanda {PLANO}`** — demanda máxima de mercado por tipo de plano

In [7]:
%%writefile bruto.mod
# ---- NeuralCloud: modelo bruto (receita bruta) ----
set DC;    # datacenters (DC1, DC2, DC3)
set PLANO; # planos GPU  (Basic, Pro, Ultra)

param receita {PLANO} >= 0;   # receita bruta por slot/dia ($)
param gpu_pw  {PLANO} >= 0;   # consumo de energia (kW/slot)
param capac   {DC}    >= 0;   # capacidade de slots por DC
param pot     {DC}    >= 0;   # capacidade de potência por DC (kW)
param dem_max {PLANO} >= 0;   # demanda máxima por plano (total, todos DCs)

var x {DC, PLANO} >= 0;

maximize z: sum {i in DC, j in PLANO} receita[j] * x[i,j];

s.t. Capac    {i in DC}:    sum {j in PLANO} x[i,j]           <= capac[i];
s.t. Potencia {i in DC}:    sum {j in PLANO} gpu_pw[j]*x[i,j] <= pot[i];
s.t. Demanda  {j in PLANO}: sum {i in DC} x[i,j]              <= dem_max[j];

Writing bruto.mod


### 2.2 Dados (`bruto.dat`)

**Receitas brutas** ($/slot/dia): Basic=\$15, Pro=\$30, Ultra=\$60  
**Potência** (kW/slot): Basic=1, Pro=2, Ultra=4  

> *Observe que receita/kW = 15 para todos os planos — o solver escolherá a mistura pelo valor de sombra dos gargalos.*

In [8]:
%%writefile bruto.dat
set DC    := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param receita :=
    Basic   15
    Pro     30
    Ultra   60 ;

param gpu_pw :=
    Basic   1
    Pro     2
    Ultra   4 ;

param capac :=
    DC1  3000
    DC2  4000
    DC3  3500 ;

param pot :=
    DC1  2800
    DC2  3600
    DC3  3200 ;

param dem_max :=
    Basic  6000
    Pro    4000
    Ultra  1500 ;

Writing bruto.dat


### 2.3 Resolver e validar (Tarefa 2.1)

In [ ]:
ampl.reset()
ampl.read("bruto.mod")
ampl.read_data("bruto.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z_bruto = ampl.get_objective("z").value()
print(f"z* (bruto) = $ {z_bruto:,.2f}")

x_bruto = ampl.get_variable("x").get_values().to_pandas()
x_bruto.columns = ["x*"]
x_bruto = x_bruto[x_bruto["x*"] > 1e-6]  # exibe apenas variáveis não-nulas
print("\nAlocação ótima (variáveis > 0):")
display(x_bruto)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 144000
5 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* (bruto) = $ 144,000.00

Alocação ótima (variáveis > 0):


x*
index0 index1      
DC1    Pro     1400
DC2    Pro      400
       Ultra    700
DC3    Ultra    800

### 2.4 Tabela de Restrições — todas as famílias

Extraímos cada família separadamente e concatenamos, ordenando por preço-sombra decrescente.  
Esse padrão é idêntico ao da Toyco — apenas o nome da família muda.

In [10]:
def extrai_restr(ampl, familia, sentido="ub"):
    """Extrai tabela de sensibilidade para uma família de restrições."""
    bound = f"{familia}.ub" if sentido == "ub" else f"{familia}.lb"
    df = ampl.get_data(
        f"{familia}.body",
        bound,
        f"{familia}.dual",
        f"{familia}.sensrhslo",
        f"{familia}.sensrhshi",
    ).to_pandas()
    df.columns = ["body", "b", "y", "rhslo", "rhshi"]
    df["folga"]   = (df["b"] - df["body"]).abs()
    df["status"]  = df["folga"].apply(lambda f: "ativa" if f < 1e-6 else "folgada")
    df["família"] = familia
    return df[["família", "b", "folga", "y", "rhslo", "rhshi", "status"]]

df_capac    = extrai_restr(ampl, "Capac")
df_potencia = extrai_restr(ampl, "Potencia")
df_demanda  = extrai_restr(ampl, "Demanda")

df_todas = pd.concat([df_capac, df_potencia, df_demanda])
df_todas = df_todas.sort_values("y", ascending=False)
print("Tabela de restrições (ordenada por y decrescente):")
display(df_todas.round(4))

Tabela de restrições (ordenada por y decrescente):


,família,b,folga,y,rhslo,rhshi,status
DC2,Potencia,3600,0,15,2800,8000,ativa
DC3,Potencia,3200,0,15,2400,6000,ativa
DC1,Potencia,2800,0,15,0,6000,ativa
DC1,Capac,3000,1600,0,1400,100000000000000000000,folgada
DC3,Capac,3500,2700,0,800,100000000000000000000,folgada
DC2,Capac,4000,2900,0,1100,100000000000000000000,folgada
Basic,Demanda,6000,6000,0,0,100000000000000000000,folgada
Pro,Demanda,4000,2200,0,1800,100000000000000000000,folgada
Ultra,Demanda,1500,0,0,800,1700,ativa


### 2.5 Tabela de Variáveis

O índice agora tem dois níveis `(DC, PLANO)` — o `.to_pandas()` já devolve um MultiIndex.

In [11]:
df_xvar = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_xvar.columns = ["x*", "rc", "objlo", "objhi"]

# Adiciona receita bruta como c_j
receita_df = ampl.get_parameter("receita").to_pandas()
receita_df.columns = ["c_j"]
# receita indexada por PLANO; precisamos alinhar com o índice (DC, PLANO)
df_xvar["c_j"] = df_xvar.index.get_level_values(-1).map(receita_df["c_j"])

df_xvar = df_xvar[["c_j", "x*", "rc", "objlo", "objhi"]]
df_xvar = df_xvar.sort_values("rc", ascending=True)  # rc negativo = não-básica
print("Tabela de variáveis (ordenada por rc):")
display(df_xvar.round(4))

Tabela de variáveis (ordenada por rc):


c_j    x*  rc                   objlo                  objhi
index0 index1                                                              
DC1    Basic    15     0   0  -100000000000000000000                     15
       Pro      30  1400   0                      30  100000000000000000000
       Ultra    60     0   0  -100000000000000000000                     60
DC2    Basic    15     0   0  -100000000000000000000                     15
       Pro      30   400   0                      30                     30
       Ultra    60   700   0                      60                     60
DC3    Basic    15     0   0  -100000000000000000000                     15
       Pro      30     0   0  -100000000000000000000                     30
       Ultra    60   800   0                      60  100000000000000000000

### 2.6 Tarefa 2.2 — O Melhor Investimento

A NeuralCloud vai ampliar a potência de **um** dos três DCs. Sem rodar o solver novamente, analise a coluna `Potencia[i]` da tabela de restrições:

- **P1. Preço-sombra.** Qual DC ganharia se olhássemos só o maior $y_i$?
- **P2. Faixa.** Quanto cada DC pode crescer antes que o $y_i$ mude ($\Delta b_i^\text{max}$)?
- **P3. Ganho.** Multiplicando $y_i \cdot \Delta b_i^\text{max}$, qual DC oferece o maior retorno financeiro viável no momento?

#### P1 — Preço-sombra: qual DC tem o maior $y_i$?

1. **Preço-sombra:** $y_\text{DC1} = y_\text{DC2} = y_\text{DC3} = 15$ \$/kW — os três DCs estão empatados; cada kW extra de potência gera \$15 adicionais em $z$, independentemente do datacenter.
2. **Variação proposta:** qualquer $\Delta b_i > 0$ em qualquer DC.
3. **Faixa de validade:** o empate em $y=15$ ocorre porque no modelo **bruto** a receita por kW é idêntica para todos os planos ($15/1 = 30/2 = 60/4 = 15$ \$/kW), tornando todos os DCs indiferentes sob a ótica do preço-sombra.

**Conclusão parcial:** olhando só $y_i$, não há como distinguir os DCs — todos valem \$15/kW. Para decidir o melhor investimento é preciso combinar o preço-sombra com a faixa de expansão disponível (P2 e P3).

#### P2 — Faixa: quanto cada DC pode crescer ($\Delta b_i^\text{max}$)?

A faixa de validade de $y_i = 15$ vai até `rhshi`. O crescimento máximo dentro dessa faixa é $\Delta b_i^\text{max} = \text{rhshi}_i - b_i$:

| DC | $b_i$ atual (kW) | `rhshi` (kW) | $\Delta b_i^\text{max}$ (kW) |
|----|-----------------|-------------|------------------------------|
| DC1 | 2 800 | 6 000 | **3 200** |
| DC2 | 3 600 | 8 000 | **4 400** |
| DC3 | 3 200 | 6 000 | **2 800** |

O DC2 tem a maior margem de expansão antes de mudar a base ótima; o DC3 tem a menor.

#### P3 — Ganho: em qual DC investir?

Multiplicando $y_i \cdot \Delta b_i^\text{max}$:

| DC | $y_i$ (\$/kW) | $\Delta b_i^\text{max}$ (kW) | Ganho máximo garantido |
|----|--------------|------------------------------|------------------------|
| DC1 | 15 | 3 200 | **\$48 000** |
| DC2 | 15 | 4 400 | **\$66 000** |
| DC3 | 15 | 2 800 | **\$42 000** |

**Decisão: investir no DC2.** Com a mesma rentabilidade por kW (\$15), o DC2 oferece a maior janela de expansão segura (4 400 kW), garantindo o maior retorno financeiro viável sem reotimização — \$66 000, contra \$48 000 do DC1 e \$42 000 do DC3.

> *Confirmação pelo código abaixo: `Melhor investimento: DC2 → ganho máximo = $66 000,00`.*

In [12]:
# Auxílio de cálculo para P2 e P3 — execute após preencher a análise acima
df_pot = df_potencia.copy()
df_pot["delta_max"] = df_pot["rhshi"] - df_pot["b"]
df_pot["ganho_max"] = df_pot["y"] * df_pot["delta_max"]
display(df_pot[["b", "y", "rhslo", "rhshi", "delta_max", "ganho_max"]].round(2))
melhor = df_pot["ganho_max"].idxmax()
print(f"\nMelhor investimento: {melhor}  →  ganho máximo = $ {df_pot.loc[melhor, 'ganho_max']:,.2f}")

,b,y,rhslo,rhshi,delta_max,ganho_max
DC1,2800,15,0,6000,3200,48000
DC2,3600,15,2800,8000,4400,66000
DC3,3200,15,2400,6000,2800,42000



Melhor investimento: DC2  →  ganho máximo = $ 66,000.00


---
## Parte 3 — NeuralCloud Líquido (Tarefa 2.3)

O modelo **líquido** embute os custos reais por slot/dia:  
- **Energia elétrica** — custo varia por DC (DCs mais caros penalizam planos que consomem mais kW)  
- **OPEX fixo** — água + depreciação por slot/dia (igual para todos)

A função objetivo passa a ser **receita líquida** $= \text{receita} - \text{custo\_elec}[i] \cdot \text{gpu\_pw}[j] - \text{opex}$.

> *Compare $z^*$ e a alocação com o modelo bruto. Observe o que acontece com o plano Basic.*

### 3.1 Modelo (`liquido.mod`)

In [13]:
%%writefile liquido.mod
# ---- NeuralCloud: modelo líquido (receita - custos reais) ----
set DC;
set PLANO;

param receita    {PLANO} >= 0;
param gpu_pw     {PLANO} >= 0;
param capac      {DC}    >= 0;
param pot        {DC}    >= 0;
param dem_max    {PLANO} >= 0;
param elec_custo {DC}    >= 0;  # custo de energia ($/kW/dia) - varia por DC
param opex               >= 0;  # OPEX fixo por slot/dia (água + depreciação)

var x {DC, PLANO} >= 0;

maximize z: sum {i in DC, j in PLANO}
    (receita[j] - gpu_pw[j] * elec_custo[i] - opex) * x[i,j];

s.t. Capac    {i in DC}:    sum {j in PLANO} x[i,j]           <= capac[i];
s.t. Potencia {i in DC}:    sum {j in PLANO} gpu_pw[j]*x[i,j] <= pot[i];
s.t. Demanda  {j in PLANO}: sum {i in DC} x[i,j]              <= dem_max[j];

Writing liquido.mod


### 3.2 Dados (`liquido.dat`)

DC1 tem a energia mais cara — isso penaliza planos que consomem mais kW, especialmente no DC1.

In [14]:
%%writefile liquido.dat
set DC    := DC1 DC2 DC3 ;
set PLANO := Basic Pro Ultra ;

param receita :=
    Basic   15
    Pro     30
    Ultra   60 ;

param gpu_pw :=
    Basic   1
    Pro     2
    Ultra   4 ;

param capac :=
    DC1  3000
    DC2  4000
    DC3  3500 ;

param pot :=
    DC1  2800
    DC2  3600
    DC3  3200 ;

param dem_max :=
    Basic  6000
    Pro    4000
    Ultra  1500 ;

# Custo de energia elétrica por DC ($/kW/dia)
param elec_custo :=
    DC1  13.50
    DC2  11.00
    DC3  12.00 ;

# OPEX fixo: água + depreciação ($/slot/dia)
param opex := 0.70 ;

Writing liquido.dat


### 3.3 Margens líquidas por (DC, Plano)

Antes de otimizar, veja a margem líquida de cada combinação para entender o que o solver vai preferir.

In [15]:
# Calcula margens líquidas sem rodar o AMPL
receitas   = {"Basic": 15, "Pro": 30, "Ultra": 60}
gpu_pw_val = {"Basic": 1,  "Pro": 2,  "Ultra": 4}
elec       = {"DC1": 13.50, "DC2": 11.00, "DC3": 12.00}
opex_val   = 0.70

rows = []
for dc in ["DC1", "DC2", "DC3"]:
    for pl in ["Basic", "Pro", "Ultra"]:
        margem = receitas[pl] - gpu_pw_val[pl] * elec[dc] - opex_val
        margem_kw = margem / gpu_pw_val[pl]
        rows.append({"DC": dc, "Plano": pl,
                     "receita": receitas[pl],
                     "custo_elec": gpu_pw_val[pl]*elec[dc],
                     "opex": opex_val,
                     "margem_liq": margem,
                     "margem/kW": margem_kw})

df_marg = pd.DataFrame(rows).set_index(["DC", "Plano"])
display(df_marg.round(2))

receita  custo_elec  opex  margem_liq  margem/kW
DC  Plano                                                  
DC1 Basic       15        13.5   0.7         0.8       0.80
    Pro         30        27.0   0.7         2.3       1.15
    Ultra       60        54.0   0.7         5.3       1.32
DC2 Basic       15        11.0   0.7         3.3       3.30
    Pro         30        22.0   0.7         7.3       3.65
    Ultra       60        44.0   0.7        15.3       3.82
DC3 Basic       15        12.0   0.7         2.3       2.30
    Pro         30        24.0   0.7         5.3       2.65
    Ultra       60        48.0   0.7        11.3       2.82

### 3.4 Resolver o modelo líquido

In [16]:
ampl.reset()
ampl.read("liquido.mod")
ampl.read_data("liquido.dat")

ampl.option["solver"] = "cplex"
ampl.option["cplex_options"] = "sens=1"
ampl.solve()

z_liq = ampl.get_objective("z").value()
print(f"z* (líquido) = $ {z_liq:,.2f}")
print(f"z* (bruto)   = $ {z_bruto:,.2f}")

x_liq = ampl.get_variable("x").get_values().to_pandas()
x_liq.columns = ["x* líquido"]
x_liq = x_liq[x_liq["x* líquido"] > 1e-6]
print("\nAlocação ótima — líquido (variáveis > 0):")
display(x_liq)

CPLEX 22.1.2:   alg:sens = 1
CPLEX 22.1.2: optimal solution; objective 25890
5 simplex iterations

suffix up OUT;
suffix down OUT;
suffix current OUT;
suffix sensobj OUT;
suffix senslbhi OUT;
suffix senslblo OUT;
suffix sensubhi OUT;
suffix sensublo OUT;
suffix sensobjhi OUT;
suffix sensobjlo OUT;
suffix sensrhshi OUT;
suffix sensrhslo OUT;
z* (líquido) = $ 25,890.00
z* (bruto)   = $ 144,000.00

Alocação ótima — líquido (variáveis > 0):


x* líquido
index0 index1            
DC1    Pro           1400
DC2    Pro            400
       Ultra          700
DC3    Ultra          800

### 3.5 Custo reduzido do plano Basic no modelo líquido

In [17]:
df_rc_liq = ampl.get_data(
    "x", "x.rc", "x.sensobjlo", "x.sensobjhi"
).to_pandas()
df_rc_liq.columns = ["x*", "rc", "objlo", "objhi"]

# Filtrar apenas as linhas do plano Basic
basic_rows = df_rc_liq.xs("Basic", level=-1) if isinstance(df_rc_liq.index, pd.MultiIndex) else \
             df_rc_liq[df_rc_liq.index.get_level_values(-1) == "Basic"]
print("Custo reduzido (rc) do plano Basic por DC:")
display(basic_rows.round(4))
print()
print("Tabela de restrições — modelo líquido:")
df_pot_liq = extrai_restr(ampl, "Potencia")
display(df_pot_liq.round(4))

Custo reduzido (rc) do plano Basic por DC:


,x*,rc,objlo,objhi
index0,,,,
DC1,0,-0.35,-100000000000000000000,1.15
DC2,0,-0.35,-100000000000000000000,3.65
DC3,0,-0.35,-100000000000000000000,2.65



Tabela de restrições — modelo líquido:


,família,b,folga,y,rhslo,rhshi,status
DC1,Potencia,2800,0,1.15,0,6000,ativa
DC2,Potencia,3600,0,3.65,2800,8000,ativa
DC3,Potencia,3200,0,2.65,2400,6000,ativa


### 3.6 Tarefa 2.3 — A Fragilidade do Ótimo

Compare $z^*$ e a alocação entre os modelos bruto e líquido. Use a tabela de variáveis (coluna `rc`) e de restrições (coluna `y`).

#### (a) O que o custo reduzido do plano Basic diz sobre seu desaparecimento da nova solução?

1. **Custo reduzido:** $\text{rc}_\text{Basic} = -0{,}35$ \$/slot em todos os DCs — Basic está fora da base ($x^* = 0$) e seu coeficiente objetivo atual é \$0,35 abaixo do mínimo para entrar.
2. **Variação necessária:** para Basic se tornar atrativo seria preciso aumentar a margem líquida em **\$0,35/slot** (p. ex., reduzir o custo de energia, aumentar o preço de venda, ou diminuir o OPEX).
3. **Faixa de otimalidade (`objhi`):** a margem líquida atual de Basic precisaria atingir \$1,15 no DC1, \$3,65 no DC2 ou \$2,65 no DC3 para entrar na solução. Atualmente ela é \$0,80, \$3,30 e \$2,30, respectivamente — todas abaixo do threshold.

**Conclusão:** no modelo líquido, Basic consome 1 kW/slot com receita de \$15, mas paga energia + OPEX que deixa margem de \$0,80–\$3,30 dependendo do DC. Esse valor é inferior à margem/kW dos planos Pro e Ultra, que disputam o mesmo recurso escasso (potência). O rc = −0,35 quantifica exatamente esse déficit de competitividade: Basic "perde" para Pro/Ultra na disputa pela potência e, por isso, some da solução ótima.

#### (b) O gargalo operacional mudou de lugar? O que isso alerta um engenheiro que projeta datacenters olhando apenas para receita bruta?

**Comparação dos preços-sombra de Potência:**

| DC | $y_\text{Potencia}$ bruto | $y_\text{Potencia}$ líquido |
|----|--------------------------|------------------------------|
| DC1 | 15 | **1,15** |
| DC2 | 15 | **3,65** |
| DC3 | 15 | **2,65** |

O gargalo não mudou de família: Potência continua sendo a restrição ativa em todos os DCs. O que mudou foi a magnitude relativa entre eles. No modelo bruto os três DCs valiam o mesmo ($y=15$); no modelo líquido, o DC2 passou a valer três vezes mais que o DC1 ($y=3{,}65$ contra $y=1{,}15$).

O motivo é o custo de energia. O DC2 tem eletricidade mais barata ($\$11{,}00/\text{kW}$), então cada kW extra liberado ali gera muito mais margem líquida do que no DC1 ($\$13{,}50/\text{kW}$). Esse custo diferenciado por datacenter é o que quebra a simetria que existia no modelo bruto.

**Alerta ao engenheiro:** projetar datacenters com base em receita bruta oculta o impacto dos custos operacionais na hierarquia de gargalos. Vendo $y=15$ igual para todos os DCs, um engenheiro poderia distribuir os investimentos de forma equilibrada quando, na prática, o DC2 entrega retorno líquido três vezes maior por kW ampliado. Ignorar os custos reais leva a subdimensionar a potência onde ela mais importa (DC2) e a superdimensionar onde o retorno é baixo (DC1).

---
**Referências:** H. Taha, *Pesquisa Operacional*, 8ª Ed. · [amplpy](https://amplpy.readthedocs.io/) · IBM CPLEX 22.1